# 4.8 Exploratory Data Analysis

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 4: Data Preprocessing & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-04-data-preprocessing-and-exploratory-data-analysis/04.08_exploratory-data-analysis.ipynb)

Companion code for Section 4.8 of the book. Run the cells in order from top to bottom.

**Example 4.8.1**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from scipy import stats

# Load dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame
df.rename(columns={'MedHouseVal': 'target'}, inplace=True)

print('=' * 60)
print('PHASE 1: DATASET STRUCTURE')
print('=' * 60)
print(f'Shape:  {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB')
print(df.dtypes.to_string())

print('\n' + '=' * 60)
print('PHASE 2: COMPLETENESS')
print('=' * 60)
print(df.isnull().sum())
print(f'Total missing: {df.isnull().sum().sum()}')

print('\n' + '=' * 60)
print('PHASE 3: UNIVARIATE ANALYSIS')
print('=' * 60)
print(df.describe().round(3))

# Skewness for each feature
print('\nSkewness per feature:')
for col in df.select_dtypes(include=np.number).columns:
    sk = stats.skew(df[col].dropna())
    flag = '← SKEWED (log-transform?)' if abs(sk) > 1 else ''
    print(f'  {col:22s}: {sk:+.3f}  {flag}')

print('\n' + '=' * 60)
print('PHASE 4: TARGET ANALYSIS')
print('=' * 60)
print(f'Target mean:   {df.target.mean():.4f}')
print(f'Target median: {df.target.median():.4f}')
print(f'Target std:    {df.target.std():.4f}')
print(f'Target range:  [{df.target.min():.2f}, {df.target.max():.2f}]')

print('\n' + '=' * 60)
print('PHASE 5: BIVARIATE - CORRELATION WITH TARGET')
print('=' * 60)
target_corr = df.corr()['target'].drop('target').sort_values(key=abs, ascending=False)
print(target_corr.round(4))

print('\n' + '=' * 60)
print('PHASE 6: FEATURE QUALITY')
print('=' * 60)
# Zero-variance features (constant - useless)
zero_var = df.var()[df.var() == 0].index.tolist()
print(f'Zero-variance features: {zero_var if zero_var else "None"}')

# Near-zero variance
low_var = df.var()[df.var() < 0.01].index.tolist()
print(f'Low-variance features (<0.01): {low_var if low_var else "None"}')

# Highly correlated feature pairs (potential redundancy)
corr_matrix = df.drop('target',axis=1).corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(c, r, upper_tri.loc[r,c])
    for c in upper_tri.columns for r in upper_tri.index
    if upper_tri.loc[r,c] > 0.85 and not pd.isna(upper_tri.loc[r,c])]
print(f'Highly correlated pairs (|r|>0.85):')
for feat1, feat2, corr in high_corr_pairs:
    print(f'  {feat1} - {feat2}: r={corr:.3f}')